<a href="https://colab.research.google.com/github/andrewi-bit/Data-Manipulation-with-Pandas/blob/main/Grouping_and_Sorting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
nolanbconaway_pitchfork_data_path = kagglehub.dataset_download('nolanbconaway/pitchfork-data')
datasnaek_chess_path = kagglehub.dataset_download('datasnaek/chess')
nasa_kepler_exoplanet_search_results_path = kagglehub.dataset_download('nasa/kepler-exoplanet-search-results')
residentmario_things_on_reddit_path = kagglehub.dataset_download('residentmario/things-on-reddit')
zynicide_wine_reviews_path = kagglehub.dataset_download('zynicide/wine-reviews')
residentmario_ramen_ratings_path = kagglehub.dataset_download('residentmario/ramen-ratings')
dansbecker_powerlifting_database_path = kagglehub.dataset_download('dansbecker/powerlifting-database')
datasnaek_youtube_new_path = kagglehub.dataset_download('datasnaek/youtube-new')
rtatman_188_million_us_wildfires_path = kagglehub.dataset_download('rtatman/188-million-us-wildfires')
jpmiller_publicassistance_path = kagglehub.dataset_download('jpmiller/publicassistance')

print('Data source import complete.')


**This notebook is an exercise in the [Pandas](https://www.kaggle.com/learn/pandas) course.  You can reference the tutorial at [this link](https://www.kaggle.com/residentmario/grouping-and-sorting).**

---


# Introduction

In these exercises we'll apply groupwise analysis to our dataset.

Run the code cell below to load the data before running the exercises.

In [1]:
import pandas as pd

!wget -q -O winemag-data-130k-v2.csv https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2019/2019-05-28/winemag-data-130k-v2.csv
reviews = pd.read_csv('winemag-data-130k-v2.csv', index_col=0)


print("Setup complete.")

Setup complete.


# Exercises

## 1.
Who are the most common wine reviewers in the dataset? Create a `Series` whose index is the `taster_twitter_handle` category from the dataset, and whose values count how many reviews each person wrote.

In [2]:
# Your code here
reviews_written = reviews.groupby('taster_twitter_handle').size()
reviews_written

,0
taster_twitter_handle,
@AnneInVino,3685
@JoeCz,5147
@bkfiona,27
@gordone_cellars,4177
@kerinokeefe,10776
@laurbuzz,1835
@mattkettmann,6332
@paulgwine,9532
@suskostrzewa,1085


## 2.
What is the best wine I can buy for a given amount of money? Create a `Series` whose index is wine prices and whose values is the maximum number of points a wine costing that much was given in a review. Sort the values by price, ascending (so that `4.0` dollars is at the top and `3300.0` dollars is at the bottom).

In [3]:
best_rating_per_price = reviews.groupby('price')['points'].max().sort_index()
best_rating_per_price


,points
price,
4.0,86
5.0,87
6.0,88
7.0,91
8.0,91
...,...
1900.0,98
2000.0,97
2013.0,91


## 3.
What are the minimum and maximum prices for each `variety` of wine? Create a `DataFrame` whose index is the `variety` category from the dataset and whose values are the `min` and `max` values thereof.

In [4]:
price_extremes = reviews.groupby('variety').price.agg([min, max])
price_extremes


/tmp/ipykernel_1258/2436081626.py:1: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  price_extremes = reviews.groupby('variety').price.agg([min, max])
/tmp/ipykernel_1258/2436081626.py:1: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  price_extremes = reviews.groupby('variety').price.agg([min, max])


,min,max
variety,,
Abouriou,15.0,75.0
Agiorgitiko,10.0,66.0
Aglianico,6.0,180.0
Aidani,27.0,27.0
Airen,8.0,10.0
...,...,...
Zinfandel,5.0,100.0
Zlahtina,13.0,16.0
Zweigelt,9.0,70.0


## 4.
What are the most expensive wine varieties? Create a variable `sorted_varieties` containing a copy of the dataframe from the previous question where varieties are sorted in descending order based on minimum price, then on maximum price (to break ties).

In [5]:
sorted_varieties = price_extremes.sort_values(by=['min', 'max'], ascending=False)
sorted_varieties

,min,max
variety,,
Ramisco,495.0,495.0
Terrantez,236.0,236.0
Francisa,160.0,160.0
Rosenmuskateller,150.0,150.0
Tinta Negra Mole,112.0,112.0
...,...,...
Roscetto,NaN,NaN
Sauvignon Blanc-Sauvignon Gris,NaN,NaN
Tempranillo-Malbec,NaN,NaN


## 5.
Create a `Series` whose index is reviewers and whose values is the average review score given out by that reviewer. Hint: you will need the `taster_name` and `points` columns.

In [6]:
reviewer_mean_ratings = reviews.groupby('taster_name').points.mean()
reviewer_mean_ratings

,points
taster_name,
Alexander Peartree,85.855422
Anna Lee C. Iijima,88.415629
Anne Krebiehl MW,90.562551
Carrie Dykes,86.395683
Christina Pickard,87.833333
Fiona Adams,86.888889
Jeff Jenssen,88.319756
Jim Gordon,88.626287
Joe Czerwinski,88.536235


Are there significant differences in the average scores assigned by the various reviewers? Run the cell below to use the `describe()` method to see a summary of the range of values.

In [7]:
reviewer_mean_ratings.describe()

,points
count,19.000000
mean,88.233026
std,1.243610
min,85.855422
25%,87.323501
50%,88.536235
75%,88.975256
max,90.562551


## 6.
What combination of countries and varieties are most common? Create a `Series` whose index is a `MultiIndex`of `{country, variety}` pairs. For example, a pinot noir produced in the US should map to `{"US", "Pinot Noir"}`. Sort the values in the `Series` in descending order based on wine count.

In [8]:
country_variety_counts = reviews.groupby(['country', 'variety']).size().sort_values(ascending=False)
country_variety_counts

country    variety                 
US         Pinot Noir                  9885
           Cabernet Sauvignon          7315
           Chardonnay                  6801
France     Bordeaux-style Red Blend    4725
Italy      Red Blend                   3624
                                       ... 
Argentina  Cabernet-Malbec                1
US         Tinta Madeira                  1
Italy      Centesimino                    1
           Catalanesca                    1
           Carmenère                      1
Length: 1612, dtype: int64

# Keep going

Move on to the [**data types and missing data**](https://www.kaggle.com/residentmario/data-types-and-missing-values).

---




*Have questions or comments? Visit the [course discussion forum](https://www.kaggle.com/learn/pandas/discussion) to chat with other learners.*